Elements of workflow adapted from https://github.com/eu-cdse/notebook-samples/blob/main/openeo/NDVI_Timeseries.ipynb

In [ ]:
import os
import json
import csv

# The CSV file containing experiment info
csv_file = "experiments/experiments_geo_orig.csv"

# The experiment name we want
desired_experiment_name = "pacific_palisades"

# Read the CSV and find the matching row
experiment_row = None
with open(csv_file, mode="r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row["experiment_name"] == desired_experiment_name:
            experiment_row = row
            break

if experiment_row is None:
    raise ValueError(f"No matching experiment found for {desired_experiment_name} in {csv_file}.")

# Extract bounding box
SPATIAL_EXTENT = {
    "west": float(experiment_row["west"]),
    "south": float(experiment_row["south"]),
    "east": float(experiment_row["east"]),
    "north": float(experiment_row["north"])
}

# Extract temporal extents
TEMPORAL_EXTENT = [experiment_row["temporal_start"], experiment_row["temporal_end"]]
TEMPORAL_EXTENT_FIRE = [experiment_row["fire_start"], experiment_row["fire_end"]]
TEMPORAL_EXTENT_PRE_FIRE = [TEMPORAL_EXTENT[0], TEMPORAL_EXTENT_FIRE[0]]
TEMPORAL_EXTENT_POST_FIRE = [TEMPORAL_EXTENT_FIRE[1], TEMPORAL_EXTENT[1]]

# Set up experiment directory
experiment_name = experiment_row["experiment_name"]
experiments_dir = "experiments"
experiment_path = os.path.join(experiments_dir, experiment_name)
os.makedirs(experiment_path, exist_ok=True)

# Build the coordinates for the bounding box
coordinates = [
    [SPATIAL_EXTENT["west"], SPATIAL_EXTENT["north"]],
    [SPATIAL_EXTENT["east"], SPATIAL_EXTENT["north"]],
    [SPATIAL_EXTENT["east"], SPATIAL_EXTENT["south"]],
    [SPATIAL_EXTENT["west"], SPATIAL_EXTENT["south"]],
    [SPATIAL_EXTENT["west"], SPATIAL_EXTENT["north"]]
]

# Construct the GeoJSON object
fields = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {
                "temporal_extent": TEMPORAL_EXTENT
            },
            "geometry": {
                "type": "Polygon",
                "coordinates": [coordinates]
            }
        }
    ]
}

# Write out fields.json
fields_path = os.path.join(experiment_path, "fields.json")
with open(fields_path, "w") as f:
    json.dump(fields, f, indent=4)

print(f"GeoJSON saved at: {fields_path}")

# Just to show what we loaded:
print("===== Loaded Experiment Parameters =====")
print(f"experiment_name = {experiment_name}")
print(f"SPATIAL_EXTENT = {SPATIAL_EXTENT}")
print(f"TEMPORAL_EXTENT = {TEMPORAL_EXTENT}")
print(f"TEMPORAL_EXTENT_FIRE = {TEMPORAL_EXTENT_FIRE}")
print(f"TEMPORAL_EXTENT_PRE_FIRE = {TEMPORAL_EXTENT_PRE_FIRE}")
print(f"TEMPORAL_EXTENT_POST_FIRE = {TEMPORAL_EXTENT_POST_FIRE}")


: 

In [ ]:
from utils.openeo_connection import get_openeo_connection

connection = get_openeo_connection()

ndvi_csv_path = os.path.join(experiment_path, "ndvi-results", "timeseries-smoothed.csv")
s5p_csv_path = os.path.join(experiment_path, "s5p-results", "s5p_fire.csv")

## Sentinel-2

In [ ]:
s2cube = connection.load_collection(
    "SENTINEL2_L2A",
    temporal_extent = TEMPORAL_EXTENT,
    bands=["B04", "B08", "SCL"]
)

### NDVI

In [ ]:
import scipy.signal
import numpy as np
import pandas as pd

# Compute NDVI
red = s2cube.band("B04")
nir = s2cube.band("B08")
ndvi = (nir - red) / (nir + red)

# Cloud mask using the SCL band
scl = s2cube.band("SCL")
mask = ~((scl == 4) | (scl == 5))

# 2D Gaussian kernel
g = scipy.signal.windows.gaussian(11, std=1.6)
kernel = np.outer(g, g)
kernel = kernel / kernel.sum()

# Morphological dilation of mask: convolution + threshold
mask = mask.apply_kernel(kernel)
mask = mask > 0.1

# Apply mask to NDVI
ndvi_masked = ndvi.mask(mask)
timeseries_masked = ndvi_masked.aggregate_spatial(geometries=fields, reducer="mean")

# Convert to Pandas & Save
timeseries_masked_df = pd.DataFrame(timeseries_masked.execute())
masked_output_csv = os.path.join(experiment_path, "timeseries_masked.csv")
timeseries_masked_df.to_csv(masked_output_csv, index=False)

print(f"Masked timeseries saved at: {masked_output_csv}")
timeseries_masked

In [ ]:
import openeo

# Define UDF for Savitzky-Golay smoothing
udf = openeo.UDF(
    """
from scipy.signal import savgol_filter
from openeo.udf import XarrayDataCube
import numpy as np
import xarray as xr

def apply_datacube(cube: XarrayDataCube, context: dict) -> XarrayDataCube:
    array = cube.get_array()
    
    # Get the number of available time points
    time_steps = array.sizes["t"]
    
    # Ensure at least 2 points for interpolation and 3 for smoothing
    if time_steps < 2:
        return cube  # Not enough points for interpolation
    if time_steps < 3:
        return XarrayDataCube(array.fillna(0))  # Fill missing with 0 if smoothing isn't possible
    
    # Fill NaN values using forward fill, then backward fill
    filled = array.ffill(dim='t').bfill(dim='t')

    # Determine a safe window length (must be odd and <= number of time points)
    window_size = min(time_steps, 5)  # Do not exceed available points
    if window_size % 2 == 0:
        window_size -= 1  # Ensure it's an odd number

    # Apply Savitzky-Golay smoothing
    smoothed_array = savgol_filter(filled.values, window_length=window_size, polyorder=2, axis=0)

    # Return smoothed data as a new XarrayDataCube
    return XarrayDataCube(xr.DataArray(smoothed_array, dims=array.dims, coords=array.coords))
    """
)

# Apply smoothing function
ndvi_smoothed = ndvi_masked.apply_dimension(code=udf, dimension="t")

# Aggregate smoothed NDVI over spatial extent
timeseries_smoothed = ndvi_smoothed.aggregate_spatial(geometries=fields, reducer="mean")

# Convert to Pandas & Save
timeseries_smoothed_df = pd.DataFrame(timeseries_smoothed.execute())
smoothed_output_csv = os.path.join(experiment_path, "timeseries_smoothed.csv")
timeseries_smoothed_df.to_csv(smoothed_output_csv, index=False)

print(f"Smoothed timeseries saved at: {smoothed_output_csv}")
timeseries_smoothed

In [ ]:
results_dir = os.path.join(experiment_path, "ndvi-results")
os.makedirs(results_dir, exist_ok=True)  
csv_path = os.path.join(results_dir, "timeseries-smoothed.csv")

if os.path.exists(csv_path):
    print(f"Skipping execution: File already exists at {csv_path}")
else:
    print("Executing batch processing...")

    # Execute batch processing
    job = timeseries_smoothed.execute_batch(
        out_format="CSV", title="Smoothed NDVI timeseries"
    )

    job.get_results().download_file(csv_path)

    print(f"Results saved at: {csv_path}")


In [ ]:
from bin.plot_timeseries import plot_ndvi_timeseries

plot_ndvi_timeseries(ndvi_csv_path, experiment_path, TEMPORAL_EXTENT_PRE_FIRE, TEMPORAL_EXTENT_FIRE, TEMPORAL_EXTENT_POST_FIRE)

## Sentinel-5p

In [ ]:
import os
import pandas as pd

# List of Sentinel-5P bands to process
s5p_bands = ["CO", "HCHO", "NO2", "O3", "SO2", "CH4"]  

results_dir = os.path.join(experiment_path, "s5p-results")
os.makedirs(results_dir, exist_ok=True) 
s5p_output_csv = os.path.join(results_dir, "s5p_fire.csv")


if os.path.exists(s5p_output_csv):
    print(f"Skipping execution: File already exists at {s5p_output_csv}")
else:
    print("Processing Sentinel-5P data...")

    # Dictionary dataFrames for each band
    band_dfs = {}

    # Function to process each band and convert to DataFrame
    def process_s5p_band(band_name):
        """Loads Sentinel-5P data for a specific band, aggregates, and returns a cleaned DataFrame."""
        print(f"Processing {band_name} data...")

        # Load Sentinel-5P L2 data for the band
        s5cube = connection.load_collection(
            "SENTINEL_5P_L2",
            temporal_extent=TEMPORAL_EXTENT,
            bands=[band_name]  # Can only load one band at a time
        )

        # Aggregate temporally (daily means)
        s5fire = s5cube.aggregate_temporal_period(reducer="mean", period="day")

        # Aggregate spatially (mean per parcel)
        s5fire = s5fire.aggregate_spatial(reducer="mean", geometries=fields)

        # Execute the processing graph and retrieve the result
        s5fire_result = s5fire.execute()

        df = pd.DataFrame(s5fire_result.items(), columns=["date", band_name])
        df["date"] = pd.to_datetime(df["date"])

        df[band_name] = df[band_name].apply(
            lambda x: x[0][0] if isinstance(x, list) and x and isinstance(x[0], list) and x[0] else None
        )

        df[band_name] = pd.to_numeric(df[band_name], errors="coerce")

        return df

    for band in s5p_bands:
        band_dfs[band] = process_s5p_band(band)

    s5p_combined_df = pd.DataFrame({"date": band_dfs[s5p_bands[0]]["date"]})  
    for band in s5p_bands:
        s5p_combined_df = pd.merge(s5p_combined_df, band_dfs[band], on="date", how="outer")

    # Save merged results as CSV
    s5p_combined_df.to_csv(s5p_output_csv, index=False)
    print(f"S5P fire data {s5p_bands} saved at: {s5p_output_csv}")


In [ ]:
from bin.plot_timeseries import plot_s5p_timeseries

plot_s5p_timeseries(s5p_csv_path, experiment_path, TEMPORAL_EXTENT_PRE_FIRE, TEMPORAL_EXTENT_FIRE, TEMPORAL_EXTENT_POST_FIRE)

## Merging

In [ ]:
pre_fire_start, pre_fire_end = map(lambda d: pd.to_datetime(d).tz_localize(None), TEMPORAL_EXTENT_PRE_FIRE)
fire_start, fire_end = map(lambda d: pd.to_datetime(d).tz_localize(None), TEMPORAL_EXTENT_FIRE)
post_fire_start, post_fire_end = map(lambda d: pd.to_datetime(d).tz_localize(None), TEMPORAL_EXTENT_POST_FIRE)


results_dir = os.path.join(experiment_path, "merged-results")
os.makedirs(results_dir, exist_ok=True)

# Load NDVI data
ndvi_df = pd.read_csv(ndvi_csv_path)
ndvi_df.rename(columns={"band_unnamed": "ndvi"}, inplace=True)

# Load Sentinel-5P data
s5p_df = pd.read_csv(s5p_csv_path)

# Convert date columns to make timezone-naive
ndvi_df["date"] = pd.to_datetime(ndvi_df["date"]).dt.tz_localize(None)
s5p_df["date"] = pd.to_datetime(s5p_df["date"]).dt.tz_localize(None)

# Merge NDVI and S5P DataFrames on the "date" column outer join
merged_df = pd.merge(ndvi_df, s5p_df, on="date", how="outer")

# Assign phase labels
def categorize_phase(date):
    if pd.isnull(date):  
        return None
    if pre_fire_start <= date < pre_fire_end:
        return "pre"
    elif fire_start <= date <= fire_end:
        return "active"
    elif post_fire_start < date <= post_fire_end:
        return "post"
    return None  

merged_df["phase"] = merged_df["date"].apply(categorize_phase)

merged_df.drop(columns=["feature_index"], inplace=True)

merged_csv_path = os.path.join(results_dir, "merged_timeseries.csv")
merged_df.to_csv(merged_csv_path, index=False)

print(f"Merged timeseries saved at: {merged_csv_path}")

merged_df

## Model

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(merged_csv_path)

numeric_columns = ["ndvi", "CO", "HCHO", "NO2", "O3", "SO2", "CH4"]

df[numeric_columns] = df[numeric_columns].interpolate(method="linear")
df[numeric_columns] = df[numeric_columns].ffill()
df[numeric_columns] = df[numeric_columns].bfill()

scaler = StandardScaler()
df[numeric_columns] = scaler.fit_transform(df[numeric_columns])

class TimeSeriesWindowDataset(Dataset):
    def __init__(self, data, window_size):
        """
        data: numpy array of shape (time_steps, num_features)
        window_size: int, the length of each time window
        """
        self.data = data
        self.window_size = window_size
    
    def __len__(self):
        return len(self.data) - self.window_size + 1
    
    def __getitem__(self, idx):
        # Extract a window of shape (window_size, num_features)
        window = self.data[idx: idx + self.window_size]
        return torch.tensor(window.T, dtype=torch.float)

data_array = df[numeric_columns].values
window_size = 10  
dataset = TimeSeriesWindowDataset(data_array, window_size)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
from model.vq_vae import VQVAE

# Model hyperparameters
input_channels = len(numeric_columns) 
hidden_dim = 256
latent_dim = 64
num_embeddings = 512
commitment_cost = 0.25

model = VQVAE(input_channels, hidden_dim, latent_dim, num_embeddings, commitment_cost)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
mse_loss = nn.MSELoss()



In [ ]:
import matplotlib.pyplot as plt

from torchsummary import summary

# Example: input size is (channels, sequence_length)
input_channels = len(numeric_columns)
time_steps = window_size  # which is 10 in your case

summary(model, input_size=(input_channels, time_steps))

class VQVAEWrapper(nn.Module):
    def __init__(self, vqvae):
        super().__init__()
        self.vqvae = vqvae

    def forward(self, x):
        x_recon, _ = self.vqvae(x)
        return x_recon


# from torchviz import make_dot

# # Create a dummy input
# x = next(iter(dataloader)).to(torch.device)  # one batch
# x = x[0].unsqueeze(0) if len(x.shape) == 2 else x  # ensure batch dim

# # Forward pass
# output, vq_loss = model(x)

# # Visualize
# dot = make_dot(output, params=dict(list(model.named_parameters())))
# dot.format = 'png'
# dot.render('vqvae_architecture')




def visualize_reconstruction(model, dataloader, device='cpu', n=5):
    model.eval()
    batch = next(iter(dataloader))
    inputs = batch.to(device)

    with torch.no_grad():
        reconstructions, _ = model(inputs)

    inputs = inputs.cpu().numpy()
    reconstructions = reconstructions.cpu().numpy()

    for i in range(min(n, inputs.shape[0])):
        plt.figure(figsize=(12, 3))
        for c in range(inputs.shape[1]):
            plt.subplot(1, inputs.shape[1], c + 1)
            plt.plot(inputs[i, c], label='Original', alpha=0.7)
            plt.plot(reconstructions[i, c], label='Reconstructed', alpha=0.7)
            plt.title(f'Channel {c}')
            plt.xticks([])
            if c == 0:
                plt.legend()
        plt.suptitle(f'Sample {i}')
        plt.tight_layout()
        plt.show()


from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def visualize_embeddings(model):
    embeddings = model.vq.embedding.weight.detach().cpu().numpy()

    tsne = TSNE(n_components=2, perplexity=30)
    embed_2d = tsne.fit_transform(embeddings)

    plt.figure(figsize=(6, 6))
    plt.scatter(embed_2d[:, 0], embed_2d[:, 1], c='blue', alpha=0.6)
    plt.title('Codebook Embeddings (t-SNE projection)')
    plt.show()

import torch

def visualize_codebook_usage(model, dataloader, device='cpu'):
    model.eval()
    all_indices = []

    with torch.no_grad():
        for batch in dataloader:
            x = batch.to(device)
            z_e = model.encoder(x)
            z_e_flat = z_e.permute(0, 2, 1).contiguous().view(-1, model.vq.embedding_dim)
            distances = (
                z_e_flat.pow(2).sum(1, keepdim=True)
                + model.vq.embedding.weight.pow(2).sum(1)
                - 2 * torch.matmul(z_e_flat, model.vq.embedding.weight.t())
            )
            encoding_indices = torch.argmin(distances, dim=1)
            all_indices.append(encoding_indices.cpu())

    all_indices = torch.cat(all_indices)
    plt.hist(all_indices.numpy(), bins=model.vq.num_embeddings)
    plt.title('Codebook Index Usage')
    plt.xlabel('Index')
    plt.ylabel('Frequency')
    plt.show()

visualize_reconstruction(model, dataloader, device='cpu', n=3)
visualize_embeddings(model)
visualize_codebook_usage(model, dataloader, device='cpu')
print(model)



In [ ]:
import matplotlib.pyplot as plt

num_epochs = 5000

loss_history = []  

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        
        # Forward pass
        x_recon, vq_loss = model(batch)
        
        # Compute losses
        recon_loss = mse_loss(x_recon, batch)
        loss = recon_loss + vq_loss
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    loss_history.append(avg_loss)
    
    #  reduce console spam
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(loss_history, label="Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("VQ-VAE Training Loss Over Epochs")
plt.legend()
plt.show()
plt.savefig(os.path.join(results_dir, "vq_vae_training_loss.png"))

In [ ]:
model.eval()
with torch.no_grad():
    # Take batch from the dataloader
    batch = next(iter(dataloader))
    x_recon, _ = model(batch)

# Plot the original and reconstructed signals for each channel of the first sample
sample_idx = 0
original = batch[sample_idx].cpu().numpy()      # shape: (input_channels, window_size)
reconstructed = x_recon[sample_idx].cpu().numpy()

plt.figure(figsize=(12, 2 * input_channels))
for i in range(input_channels):
    plt.subplot(input_channels, 1, i+1)
    plt.plot(original[i], label="Original")
    plt.plot(reconstructed[i], label="Reconstructed", linestyle="--")
    plt.title(numeric_columns[i])
    plt.legend()
plt.tight_layout()
plt.show()

plt.savefig(os.path.join(experiment_path, "vq_vae_reconstruction.png"))

## Prediction

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt


# 1. Load & Preprocess Data

numeric_columns = ["ndvi", "CO", "HCHO", "NO2", "O3", "SO2", "CH4"]
df[numeric_columns] = df[numeric_columns].interpolate(method="linear")
df[numeric_columns] = df[numeric_columns].ffill()
df[numeric_columns] = df[numeric_columns].bfill()

# Filter out unexpected phases, keep only pre/active/post
df = df[df["phase"].isin(["pre", "active", "post"])]

# Encode phase as integer labels
le = LabelEncoder()
df["phase_label"] = le.fit_transform(df["phase"])  # 0,1,2 => pre, active, post

# 2. Create a Windowed Dataset

class PhaseWindowDataset(Dataset):
    def __init__(self, df, numeric_cols, window_size=10):
        """
        df: pandas DataFrame containing numeric_cols and 'phase_label'
        numeric_cols: list of numeric feature names
        window_size: int, length of each time window
        """
        self.df = df.reset_index(drop=True)
        self.numeric_cols = numeric_cols
        self.window_size = window_size
        
        self.values = self.df[numeric_cols].values
        self.phases = self.df["phase_label"].values
        
    def __len__(self):
        return len(self.df) - self.window_size + 1
    
    def __getitem__(self, idx):
        window_data = self.values[idx : idx + self.window_size]
        # Label is from the last timestep in this window
        label = self.phases[idx + self.window_size - 1]
        
        # For 1D CNN: (num_features, window_size)
        window_data = torch.tensor(window_data.T, dtype=torch.float)
        label = torch.tensor(label, dtype=torch.long)
        
        return window_data, label

dataset = PhaseWindowDataset(df, numeric_columns, window_size=10)

# 3. Define Model & Helpers

class CNNPhaseClassifier(nn.Module):
    def __init__(self, input_channels, num_classes):
        super(CNNPhaseClassifier, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)  # global average pooling
        self.relu = nn.ReLU()
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # x: (batch_size, input_channels, window_size)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)        
        x = x.squeeze(-1)       
        x = self.fc(x)         
        return x

def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    for batch_data, batch_labels in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_data)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_model(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch_data, batch_labels in dataloader:
            outputs = model(batch_data)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_labels.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

def plot_confusion_and_report(true_labels, pred_labels, label_encoder, title=""):
    """
    Plots confusion matrix and prints classification report for all 3 classes
    (pre, active, post), even if some classes are missing in the test set.
    """
    all_labels = [0, 1, 2]
    cm = confusion_matrix(true_labels, pred_labels, labels=all_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f"Confusion Matrix: {title}")
    plt.show()
    
    print(f"\nClassification Report: {title}")
    print(classification_report(
        true_labels,
        pred_labels,
        labels=all_labels,
        target_names=label_encoder.classes_
    ))

# 4a. K-Fold Cross Validation
def cross_validate_model(dataset, k=5, epochs=10):
    from sklearn.model_selection import StratifiedKFold
    
    X_indices = np.arange(len(dataset))
    y_labels = []
    for idx in range(len(dataset)):
        _, label = dataset[idx]
        y_labels.append(label.item())
    y_labels = np.array(y_labels)
    
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    
    fold_accuracies = []
    fold_f1_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_indices, y_labels)):
        print(f"\n=== K-Fold: Fold {fold+1}/{k} ===")
        
        train_subset = Subset(dataset, train_idx)
        val_subset   = Subset(dataset, val_idx)
        
        train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
        val_loader   = DataLoader(val_subset, batch_size=32, shuffle=False)
        
        # New model each fold
        model = CNNPhaseClassifier(input_channels=len(numeric_columns), num_classes=3)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        
        # Train
        for epoch in range(epochs):
            loss = train_one_epoch(model, train_loader, criterion, optimizer)
        
        # Evaluate
        val_labels, val_preds = evaluate_model(model, val_loader)
        acc = accuracy_score(val_labels, val_preds)
        f1 = f1_score(val_labels, val_preds, average="weighted")
        
        print(f"Fold {fold+1} - Accuracy: {acc:.4f}, Weighted F1: {f1:.4f}")
        
        fold_accuracies.append(acc)
        fold_f1_scores.append(f1)
        
        plot_confusion_and_report(val_labels, val_preds, le, title=f"K-Fold {fold+1}")
    
    # Print average metrics across folds
    print(f"\n=== Cross-Validation ({k}-fold) Summary ===")
    print(f"Average Accuracy: {np.mean(fold_accuracies):.4f}")
    print(f"Average Weighted F1: {np.mean(fold_f1_scores):.4f}")

# 4b. Time-Based Train/Test Split
def time_based_split(dataset, train_ratio=0.8, epochs=10):
    print("\n=== Time-Based Split ===")
    
    total_len = len(dataset)
    split_index = int(train_ratio * total_len)
    
    train_subset = Subset(dataset, range(split_index))
    test_subset  = Subset(dataset, range(split_index, total_len))
    
    train_loader = DataLoader(train_subset, batch_size=32, shuffle=False)
    test_loader  = DataLoader(test_subset, batch_size=32, shuffle=False)
    
    model = CNNPhaseClassifier(input_channels=len(numeric_columns), num_classes=3)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    # Train
    for epoch in range(epochs):
        loss = train_one_epoch(model, train_loader, criterion, optimizer)
    
    # Evaluate
    test_labels, test_preds = evaluate_model(model, test_loader)
    acc = accuracy_score(test_labels, test_preds)
    f1 = f1_score(test_labels, test_preds, average="weighted")
    
    print(f"Time-based Test - Accuracy: {acc:.4f}, Weighted F1: {f1:.4f}")
    
    # Confusion matrix + classification report
    plot_confusion_and_report(test_labels, test_preds, le, title="Time-based Split")

# 5. Run All Approaches
if __name__ == "__main__":
    # 1) K-Fold Cross Validation
    cross_validate_model(dataset, k=5, epochs=10)
    
    # 2) Time-Based Train/Test Split
    time_based_split(dataset, train_ratio=0.8, epochs=10)
